# ✈️ Airline Fare Intelligence & Buy-Now-or-Wait Advisor
**Adapted for `Clean_Dataset.csv` (300,153 Domestic Flight Observations)**  
**Project Track:** A - Regression | **Domain:** Aviation / Travel  

### Dataset Suitability Summary:
- **Suitable for Core Pipeline:** `Clean_Dataset.csv` tracks **1,561 unique flights** across **6 Indian metro cities** (Delhi, Mumbai, Bangalore, Kolkata, Hyderabad, Chennai), **6 airlines**, **2 classes** (Economy & Business), and **49 days** of booking lead times (`days_left` from 1 to 49).
- **Exact Self-Join Capability:** Each `(flight, source_city, destination_city, class)` trajectory is tracked day-by-day, allowing us to compute the exact fare **7 days later** (`days_left - 7`) and **7 days earlier** (`days_left + 7`) to train both the **Fair Fare Regressor** and the **7-Day >5% Rise Classifier**, as well as run a full **Buy-Now-or-Wait Backtest**.
- **Important Adaptations:** Because `Clean_Dataset.csv` is already standardized (no dirty spellings/typos) and does not contain calendar `departure_date` columns, calendar/festival date extraction and chronological date splits are replaced by **Group-based (Flight-ID) Train / Validation / Test splits** so zero flight leakage occurs between splits.


## 1. Environment Setup & Project Structure


In [ ]:
!pip install -q lightgbm xgboost optuna shap mlflow streamlit pytest

import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_percentage_error, mean_absolute_error, r2_score,
    roc_auc_score, precision_recall_curve, auc, confusion_matrix,
    precision_score, recall_score
)
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
import lightgbm as lgb
import xgboost as xgb
import optuna
import shap
import mlflow

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid")

for d in ["data/raw", "data/processed", "src", "app", "tests", "outputs/figures", "outputs/reports", "models"]:
    os.makedirs(d, exist_ok=True)

print("✅ Environment and directories ready!")


## 2. Load & Inspect `Clean_Dataset.csv`
Upload `Clean_Dataset.csv` to the Colab root directory (or `data/raw/Clean_Dataset.csv`).


In [ ]:
DATA_PATH = "Clean_Dataset.csv" if os.path.exists("Clean_Dataset.csv") else "data/raw/Clean_Dataset.csv"
df_raw = pd.read_csv(DATA_PATH)
if "Unnamed: 0" in df_raw.columns:
    df_raw = df_raw.drop(columns=["Unnamed: 0"])

print(f"Loaded {len(df_raw):,} rows x {df_raw.shape[1]} columns")
df_raw.head()


## 3. Data Cleaning, Normalization & Self-Join Fare History (`src/features.py`)


In [ ]:
%%writefile src/features.py
import numpy as np
import pandas as pd

STOPS_MAP = {"zero": 0, "one": 1, "two_or_more": 2}
TIME_ORDER = {
    "Early_Morning": 0,
    "Morning": 1,
    "Afternoon": 2,
    "Evening": 3,
    "Night": 4,
    "Late_Night": 5
}

def clean_and_engineer(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
        
    # Standardize columns to match project conventions
    df = df.rename(columns={
        "flight": "flight_no",
        "source_city": "origin",
        "destination_city": "destination",
        "class": "travel_class",
        "days_left": "days_to_departure",
        "price": "fare_inr"
    })
    
    # Deduplicate exact flight-route-class-days_to_departure snapshots (keep cheapest fare logged)
    df = df.sort_values("fare_inr").drop_duplicates(
        subset=["airline", "flight_no", "origin", "destination", "travel_class", "days_to_departure"],
        keep="first"
    ).reset_index(drop=True)
    
    # Numeric encodings
    df["stops_num"] = df["stops"].map(STOPS_MAP).fillna(0).astype(int)
    df["dep_time_idx"] = df["departure_time"].map(TIME_ORDER).fillna(2).astype(int)
    df["arr_time_idx"] = df["arrival_time"].map(TIME_ORDER).fillna(2).astype(int)
    df["duration_mins"] = (df["duration"] * 60).round().astype(int)
    df["is_red_eye"] = df["departure_time"].isin(["Late_Night", "Early_Morning"]).astype(int)
    df["is_peak_evening"] = df["departure_time"].isin(["Evening", "Night"]).astype(int)
    df["route"] = df["origin"] + " - " + df["destination"]
    
    # Self-join on (flight_no, origin, destination, travel_class) for 7-day lead and lag
    lookup = df[["flight_no", "origin", "destination", "travel_class", "days_to_departure", "fare_inr"]].copy()
    
    # 1 week later means days_to_departure - 7
    df["days_minus_7"] = df["days_to_departure"] - 7
    df = df.merge(
        lookup.rename(columns={"days_to_departure": "days_minus_7", "fare_inr": "fare_next_week"}),
        on=["flight_no", "origin", "destination", "travel_class", "days_minus_7"],
        how="left"
    )
    
    # 1 week earlier means days_to_departure + 7
    df["days_plus_7"] = df["days_to_departure"] + 7
    df = df.merge(
        lookup.rename(columns={"days_to_departure": "days_plus_7", "fare_inr": "fare_prev_week"}),
        on=["flight_no", "origin", "destination", "travel_class", "days_plus_7"],
        how="left"
    )
    
    df["lag_fare_change_pct"] = ((df["fare_inr"] - df["fare_prev_week"]) / df["fare_prev_week"]).fillna(0.0)
    df["rise_5pct"] = np.where(
        df["fare_next_week"].notna(),
        (df["fare_next_week"] > 1.05 * df["fare_inr"]).astype(int),
        np.nan
    )
    
    df = df.drop(columns=["days_minus_7", "days_plus_7"])
    return df


In [ ]:
from src.features import clean_and_engineer

df_feat = clean_and_engineer(df_raw)
df_feat.to_csv("data/processed/fares_cleaned.csv", index=False)

# Leakage-free split grouped by flight_no (70% Train, 15% Val, 15% Test)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(df_feat, groups=df_feat["flight_no"]))
train_df = df_feat.iloc[train_idx].copy().reset_index(drop=True)
temp_df  = df_feat.iloc[temp_idx].copy().reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["flight_no"]))
val_df  = temp_df.iloc[val_idx].copy().reset_index(drop=True)
test_df = temp_df.iloc[test_idx].copy().reset_index(drop=True)

print(f"Deduplicated & Cleaned Observations: {len(df_feat):,}")
print(f"Train Set: {len(train_df):,} rows ({train_df['flight_no'].nunique()} flights)")
print(f"Val Set  : {len(val_df):,} rows ({val_df['flight_no'].nunique()} flights)")
print(f"Test Set : {len(test_df):,} rows ({test_df['flight_no'].nunique()} flights)")


## 4. Unit Tests (`tests/test_features.py`)


In [ ]:
%%writefile tests/test_features.py
import pandas as pd
from src.features import clean_and_engineer

def test_clean_and_self_join():
    sample = pd.DataFrame([
        {"airline": "Indigo", "flight": "6E-101", "source_city": "Delhi", "departure_time": "Evening",
         "stops": "zero", "arrival_time": "Night", "destination_city": "Mumbai", "class": "Economy",
         "duration": 2.15, "days_left": 21, "price": 5000},
        {"airline": "Indigo", "flight": "6E-101", "source_city": "Delhi", "departure_time": "Evening",
         "stops": "zero", "arrival_time": "Night", "destination_city": "Mumbai", "class": "Economy",
         "duration": 2.15, "days_left": 14, "price": 5600},
    ])
    out = clean_and_engineer(sample)
    row_21 = out[out["days_to_departure"] == 21].iloc[0]
    assert row_21["fare_next_week"] == 5600
    assert row_21["rise_5pct"] == 1
    assert row_21["stops_num"] == 0


In [ ]:
!pytest -q


## 5. Exploratory Data Analysis (Key Project Charts)


In [ ]:
econ = df_feat[df_feat["travel_class"] == "Economy"].copy()

# Chart 1: Fare Distribution by Class (Log Scale)
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_feat, x="travel_class", y="fare_inr", palette="Set2")
plt.yscale("log")
plt.title("Chart 1: Fare Distribution by Class (Log Scale)")
plt.ylabel("Fare (INR, log scale)")
plt.tight_layout()
plt.savefig("outputs/figures/chart01_fare_dist_class.png", dpi=150)
plt.show()

# Chart 2: Fare Curve by Days to Departure (Top 8 Routes)
top_routes = econ["route"].value_counts().head(8).index
econ_top = econ[econ["route"].isin(top_routes)].copy()
route_med = econ_top.groupby("route")["fare_inr"].transform("median")
econ_top["fare_index"] = (econ_top["fare_inr"] / route_med) * 100

g = sns.FacetGrid(econ_top, col="route", col_wrap=4, height=3, aspect=1.2)
g.map_dataframe(sns.lineplot, x="days_to_departure", y="fare_index", estimator="median", errorbar=("pi", 50))
g.set_titles("{col_name}")
g.set_axis_labels("Days Left", "Fare Index (100 = Median)")
plt.subplots_adjust(top=0.88)
g.fig.suptitle("Chart 2: Fare Curve by Days to Departure per Route")
plt.savefig("outputs/figures/chart02_route_fare_curves.png", dpi=150)
plt.show()

# Chart 3 & 4: Airline Fare Levels & Departure-Time x Stops Heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
nonstop_econ = econ[econ["stops_num"] == 0]
order_air = nonstop_econ.groupby("airline")["fare_inr"].median().sort_values().index
sns.barplot(data=nonstop_econ, x="airline", y="fare_inr", order=order_air, estimator=np.median, ax=axes[0], palette="viridis")
axes[0].set_title("Chart 3: Median Non-Stop Economy Fare by Airline")
axes[0].set_ylabel("Median Fare (INR)")

piv = econ.pivot_table(index="departure_time", columns="stops", values="fare_inr", aggfunc="median")
sns.heatmap(piv, annot=True, fmt=".0f", cmap="YlOrRd", ax=axes[1])
axes[1].set_title("Chart 4: Median Economy Fare by Departure Time x Stops")
plt.tight_layout()
plt.savefig("outputs/figures/chart03_04_airlines_heatmap.png", dpi=150)
plt.show()

# Chart 7: Frequency of >5% Fare Rise Within 7 Days by Weeks Out
valid_rise = econ.dropna(subset=["rise_5pct"]).copy()
valid_rise["week_out"] = (valid_rise["days_to_departure"] // 7)
rise_rates = valid_rise.groupby("week_out")["rise_5pct"].mean() * 100
plt.figure(figsize=(8, 4))
rise_rates.plot(kind="bar", color="coral")
plt.title("Chart 7: % Chance Economy Fare Rises >5% in 7 Days by Weeks Out")
plt.xlabel("Weeks Before Departure")
plt.ylabel("Probability of >5% Rise (%)")
plt.tight_layout()
plt.savefig("outputs/figures/chart07_rise_rate_by_week.png", dpi=150)
plt.show()


## 6. Milestone 3: Fair Fare Regression, Optuna Tuning & SHAP


In [ ]:
mlflow.set_experiment("Airline_Fare_Intelligence_CleanDataset")

FEATURES = [
    "days_to_departure", "stops_num", "duration_mins", "dep_time_idx",
    "arr_time_idx", "is_red_eye", "is_peak_evening",
    "airline", "origin", "destination", "travel_class"
]
CAT_COLS = ["airline", "origin", "destination", "travel_class"]

for df_part in [train_df, val_df, test_df]:
    for c in CAT_COLS:
        df_part[c] = df_part[c].astype("category")

# 1. Lookup Baseline (Route x Class x Days_to_Departure)
lookup_tbl = train_df.groupby(["origin", "destination", "travel_class", "days_to_departure"])["fare_inr"].median().reset_index()
lookup_tbl.rename(columns={"fare_inr": "lookup_pred"}, inplace=True)
val_lookup = val_df.merge(lookup_tbl, on=["origin", "destination", "travel_class", "days_to_departure"], how="left")["lookup_pred"].fillna(train_df["fare_inr"].median())
baseline_val_mape = mean_absolute_percentage_error(val_df["fare_inr"], val_lookup)

# One-hot matrices for Ridge & Random Forest
X_tr_ohe = pd.get_dummies(train_df[FEATURES], drop_first=True)
X_va_ohe = pd.get_dummies(val_df[FEATURES], drop_first=True).reindex(columns=X_tr_ohe.columns, fill_value=0)
y_tr_log = np.log(train_df["fare_inr"])

results_reg = {"Lookup Table": baseline_val_mape}

# 2. Ridge
ridge = Ridge(alpha=1.0).fit(X_tr_ohe, y_tr_log)
results_reg["Ridge"] = mean_absolute_percentage_error(val_df["fare_inr"], np.exp(ridge.predict(X_va_ohe)))

# 3. Random Forest (sampled 60k rows for fast Colab runtime)
rf_sample = train_df.sample(min(60000, len(train_df)), random_state=42)
X_rf_ohe = pd.get_dummies(rf_sample[FEATURES], drop_first=True).reindex(columns=X_tr_ohe.columns, fill_value=0)
rf = RandomForestRegressor(n_estimators=80, max_depth=14, random_state=42, n_jobs=-1).fit(X_rf_ohe, np.log(rf_sample["fare_inr"]))
results_reg["Random Forest"] = mean_absolute_percentage_error(val_df["fare_inr"], np.exp(rf.predict(X_va_ohe)))

# 4. XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=250, learning_rate=0.06, max_depth=7, enable_categorical=True, random_state=42, n_jobs=-1)
xgb_model.fit(train_df[FEATURES], y_tr_log)
results_reg["XGBoost"] = mean_absolute_percentage_error(val_df["fare_inr"], np.exp(xgb_model.predict(val_df[FEATURES])))

# 5. Default LightGBM
lgb_def = lgb.LGBMRegressor(n_estimators=250, learning_rate=0.06, random_state=42, verbose=-1, n_jobs=-1)
lgb_def.fit(train_df[FEATURES], y_tr_log)
results_reg["LightGBM (Default)"] = mean_absolute_percentage_error(val_df["fare_inr"], np.exp(lgb_def.predict(val_df[FEATURES])))

# 6. Optuna Tuning for LightGBM (25 trials)
def objective(trial):
    params = {
        "objective": "regression",
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 127),
        "max_depth": trial.suggest_int("max_depth", 5, 14),
        "min_child_samples": trial.suggest_int("min_child_samples", 15, 60),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "n_estimators": 300,
        "random_state": 42,
        "verbose": -1,
        "n_jobs": -1
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(train_df[FEATURES], y_tr_log)
    preds = np.exp(m.predict(val_df[FEATURES]))
    return mean_absolute_percentage_error(val_df["fare_inr"], preds)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=25)
results_reg["LightGBM (Tuned)"] = study.best_value

for k, v in results_reg.items():
    print(f"{k:22s} -> Val MAPE: {v*100:.2f}%")

# Refit Tuned LightGBM on Train + Val
best_params = {**study.best_params, "objective": "regression", "n_estimators": 350, "random_state": 42, "verbose": -1, "n_jobs": -1}
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
fare_model = lgb.LGBMRegressor(**best_params)
fare_model.fit(train_val_df[FEATURES], np.log(train_val_df["fare_inr"]))

# Evaluate on Test Set
test_preds = np.exp(fare_model.predict(test_df[FEATURES]))
test_mape = mean_absolute_percentage_error(test_df["fare_inr"], test_preds)
test_mae = mean_absolute_error(test_df["fare_inr"], test_preds)
test_r2 = r2_score(np.log(test_df["fare_inr"]), np.log(test_preds))

print(f"\n✅ Tuned Fare Model Test Metrics -> MAPE: {test_mape*100:.2f}% | MAE: ₹{test_mae:.0f} | R²: {test_r2:.3f}")
joblib.dump(fare_model, "models/best_fare_lgb.pkl")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Chart 8: Optuna History
trial_vals = [t.value * 100 for t in study.trials]
axes[0].plot(trial_vals, marker="o", alpha=0.5, label="Trial MAPE")
axes[0].plot(np.minimum.accumulate(trial_vals), color="red", linewidth=2, label="Best MAPE")
axes[0].set_title("Chart 8: Optuna Optimisation History (Val MAPE %)")
axes[0].set_xlabel("Trial"); axes[0].set_ylabel("MAPE (%)"); axes[0].legend()

# Chart 9: Model Comparison
comp_s = pd.Series({k: v*100 for k, v in results_reg.items()}).sort_values(ascending=False)
comp_s.plot(kind="barh", ax=axes[1], color="steelblue")
axes[1].set_title("Chart 9: Fare Model Validation MAPE (%) Comparison")
axes[1].set_xlabel("Validation MAPE (%)")
plt.tight_layout()
plt.savefig("outputs/figures/chart08_09_optuna_comparison.png", dpi=150)
plt.show()

# Chart 10: Actual vs Predicted
plt.figure(figsize=(6, 5))
sample_idx = np.random.choice(len(test_df), min(5000, len(test_df)), replace=False)
plt.scatter(test_df["fare_inr"].iloc[sample_idx], test_preds[sample_idx], alpha=0.25, s=8)
lims = [test_df["fare_inr"].min(), test_df["fare_inr"].max()]
plt.plot(lims, lims, "r--")
plt.xscale("log"); plt.yscale("log")
plt.title(f"Chart 10: Test Actual vs Predicted Fare (MAPE={test_mape*100:.1f}%, R²={test_r2:.2f})")
plt.xlabel("Actual Fare (INR)"); plt.ylabel("Predicted Fare (INR)")
plt.tight_layout()
plt.savefig("outputs/figures/chart10_actual_vs_pred.png", dpi=150)
plt.show()

# Chart 11: SHAP Summary
explainer = shap.TreeExplainer(fare_model)
sample_X = test_df[FEATURES].sample(min(1500, len(test_df)), random_state=42)
shap_values = explainer.shap_values(sample_X)
plt.figure(figsize=(9, 5))
shap.summary_plot(shap_values, sample_X, show=False)
plt.title("Chart 11: SHAP Feature Importance Summary")
plt.tight_layout()
plt.savefig("outputs/figures/chart11_shap_summary.png", dpi=150)
plt.show()


## 7. Milestone 4: Out-of-Fold Stacking & 7-Day Rise Classifier


In [ ]:
def attach_expected_next_week_fare(df_in, reg_model, oof_pred_col=None):
    df_out = df_in.copy()
    if oof_pred_col is not None:
        df_out["pred_fair_fare"] = oof_pred_col
    else:
        df_out["pred_fair_fare"] = np.exp(reg_model.predict(df_out[FEATURES]))
        
    # Simulate same flight 7 days closer to departure
    df_next = df_out[FEATURES].copy()
    df_next["days_to_departure"] = (df_next["days_to_departure"] - 7).clip(lower=1)
    df_out["pred_next_week_fare"] = np.exp(reg_model.predict(df_next))
    
    # Stacked features
    df_out["fare_ratio_to_fair"] = df_out["fare_inr"] / df_out["pred_fair_fare"]
    df_out["expected_move_pct"] = (df_out["pred_next_week_fare"] - df_out["fare_inr"]) / df_out["fare_inr"]
    return df_out

# GroupKFold Out-of-Fold predictions on train_df
gkf = GroupKFold(n_splits=5)
oof_fair_fares = np.zeros(len(train_df))
for tr_idx, va_idx in gkf.split(train_df, groups=train_df["flight_no"]):
    m_fold = lgb.LGBMRegressor(**best_params)
    m_fold.fit(train_df.iloc[tr_idx][FEATURES], np.log(train_df.iloc[tr_idx]["fare_inr"]))
    oof_fair_fares[va_idx] = np.exp(m_fold.predict(train_df.iloc[va_idx][FEATURES]))

train_clf_df = attach_expected_next_week_fare(train_df, fare_model, oof_pred_col=oof_fair_fares).dropna(subset=["rise_5pct"])
val_clf_df   = attach_expected_next_week_fare(val_df, fare_model).dropna(subset=["rise_5pct"])
test_clf_df  = attach_expected_next_week_fare(test_df, fare_model).dropna(subset=["rise_5pct"])

CLF_FEATURES = [
    "days_to_departure", "fare_inr", "pred_fair_fare", "pred_next_week_fare",
    "fare_ratio_to_fair", "expected_move_pct", "lag_fare_change_pct",
    "stops_num", "dep_time_idx", "arr_time_idx", "is_red_eye", "is_peak_evening",
    "airline", "origin", "destination", "travel_class"
]

# LightGBM Classifier
clf_lgb = lgb.LGBMClassifier(n_estimators=250, learning_rate=0.04, num_leaves=31, random_state=42, verbose=-1, n_jobs=-1)
clf_lgb.fit(train_clf_df[CLF_FEATURES], train_clf_df["rise_5pct"])

# Logistic Regression Baseline
X_tr_lr = pd.get_dummies(train_clf_df[CLF_FEATURES], drop_first=True).fillna(0)
X_te_lr = pd.get_dummies(test_clf_df[CLF_FEATURES], drop_first=True).reindex(columns=X_tr_lr.columns, fill_value=0)
clf_lr = LogisticRegression(max_iter=1000).fit(X_tr_lr, train_clf_df["rise_5pct"])

test_probs_lgb = clf_lgb.predict_proba(test_clf_df[CLF_FEATURES])[:, 1]
test_probs_lr  = clf_lr.predict_proba(X_te_lr)[:, 1]
y_test_clf = test_clf_df["rise_5pct"].values

roc_lgb = roc_auc_score(y_test_clf, test_probs_lgb)
prec_lgb, rec_lgb, _ = precision_recall_curve(y_test_clf, test_probs_lgb)
pr_auc_lgb = auc(rec_lgb, prec_lgb)
roc_lr = roc_auc_score(y_test_clf, test_probs_lr)

print(f"✅ LightGBM Rise Classifier -> Test ROC-AUC: {roc_lgb:.3f} | PR-AUC: {pr_auc_lgb:.3f}")
print(f"   Logistic Regression      -> Test ROC-AUC: {roc_lr:.3f}")

joblib.dump(clf_lgb, "models/best_rise_classifier.pkl")

# Chart 14 & 15: PR Curve & Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(rec_lgb, prec_lgb, label=f"LightGBM (PR-AUC={pr_auc_lgb:.2f}, ROC-AUC={roc_lgb:.2f})")
axes[0].set_title("Chart 14: Test Precision-Recall Curve")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision"); axes[0].legend()

preds_05 = (test_probs_lgb >= 0.5).astype(int)
cm = confusion_matrix(y_test_clf, preds_05)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1])
axes[1].set_title(f"Chart 15: Test Confusion Matrix (Prec={precision_score(y_test_clf, preds_05):.2f}, Rec={recall_score(y_test_clf, preds_05):.2f})")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")
plt.tight_layout()
plt.savefig("outputs/figures/chart14_15_classifier_eval.png", dpi=150)
plt.show()


## 8. Milestone 5: Buy-Now-or-Wait Advisor & Backtest


In [ ]:
def evaluate_policy(df_eval, wait_mask):
    actual_paid = np.where(wait_mask, df_eval["fare_next_week"], df_eval["fare_inr"])
    saved_vs_buy_now = df_eval["fare_inr"] - actual_paid
    regret = np.where(wait_mask, df_eval["fare_next_week"] > df_eval["fare_inr"], df_eval["fare_next_week"] < df_eval["fare_inr"])
    return saved_vs_buy_now.mean(), regret.mean() * 100

# Tune threshold tau on Validation set
val_clf_df["prob_rise"] = clf_lgb.predict_proba(val_clf_df[CLF_FEATURES])[:, 1]
taus = np.linspace(0.05, 0.55, 26)
val_savings, val_regrets = [], []

for t in taus:
    wait_cond = (val_clf_df["prob_rise"] < t) & (val_clf_df["pred_next_week_fare"] < val_clf_df["fare_inr"])
    s, r = evaluate_policy(val_clf_df, wait_cond)
    val_savings.append(s)
    val_regrets.append(r)

best_tau = taus[np.argmax(val_savings)]
print(f"✅ Optimal Validation Threshold tau = {best_tau:.2f} (Val Avg Saved: ₹{max(val_savings):.1f})")

# Chart 13: Threshold Tuning
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(taus, val_savings, color="green", marker="o", label="Avg Saved per Search (₹)")
ax1.set_xlabel("Threshold tau"); ax1.set_ylabel("Avg Saved (₹)", color="green")
ax2 = ax1.twinx()
ax2.plot(taus, val_regrets, color="red", linestyle="--", label="Regret Rate (%)")
ax2.set_ylabel("Regret Rate (%)", color="red")
plt.title("Chart 13: Threshold Tuning on Validation Set")
plt.tight_layout()
plt.savefig("outputs/figures/chart13_threshold_tuning.png", dpi=150)
plt.show()

# Backtest all strategies on Test Set
test_clf_df["prob_rise"] = test_probs_lgb

strategies = {
    "Always buy now": np.zeros(len(test_clf_df), dtype=bool),
    "Always wait a week": np.ones(len(test_clf_df), dtype=bool),
    "Fare model only": test_clf_df["pred_next_week_fare"] < test_clf_df["fare_inr"],
    "Rise classifier only": test_clf_df["prob_rise"] < best_tau,
    "Combined advisor": (test_clf_df["prob_rise"] < best_tau) & (test_clf_df["pred_next_week_fare"] < test_clf_df["fare_inr"]),
    "Oracle (perfect foresight)": test_clf_df["fare_next_week"] < test_clf_df["fare_inr"]
}

bt_rows = []
for name, mask in strategies.items():
    avg_saved, regret_pct = evaluate_policy(test_clf_df, mask)
    bt_rows.append({"Strategy": name, "Avg_Saved_INR": round(avg_saved, 2), "Regret_Rate_Pct": round(regret_pct, 1)})

bt_df = pd.DataFrame(bt_rows)
print("\n📊 Test Set Backtest Results:")
display(bt_df)

# Chart 16: Backtest Bar Chart
plt.figure(figsize=(9, 4.5))
colors = ["gray", "salmon", "lightcoral", "skyblue", "dodgerblue", "dimgray"]
bars = plt.barh(bt_df["Strategy"][::-1], bt_df["Avg_Saved_INR"][::-1], color=colors[::-1])
plt.axvline(0, color="black", linewidth=0.8)
for bar, (_, row) in zip(bars, bt_df[::-1].iterrows()):
    w = bar.get_width()
    offset = 10 if w >= 0 else -140
    plt.text(w + offset, bar.get_y() + bar.get_height()/2, f"₹{row['Avg_Saved_INR']:.0f} (regret {row['Regret_Rate_Pct']:.0f}%)", va="center", fontsize=9)
plt.title("Chart 16: Backtest — Average ₹ Saved per Search vs Always Buying Now")
plt.xlabel("Average ₹ Saved per Search")
plt.xlim(min(bt_df["Avg_Saved_INR"].min() - 200, -150), bt_df["Avg_Saved_INR"].max() + 200)
plt.tight_layout()
plt.savefig("outputs/figures/chart16_backtest_strategies.png", dpi=150)
plt.show()

# Save Reports
bt_df.to_csv("outputs/reports/metrics.csv", index=False)
summary = {
    "test_fare_mape_pct": round(test_mape * 100, 2),
    "test_fare_r2": round(test_r2, 3),
    "test_classifier_roc_auc": round(roc_lgb, 3),
    "optimal_tau": round(float(best_tau), 3),
    "advisor_avg_saved_inr": float(bt_df.loc[bt_df["Strategy"]=="Combined advisor", "Avg_Saved_INR"].values[0]),
    "advisor_regret_pct": float(bt_df.loc[bt_df["Strategy"]=="Combined advisor", "Regret_Rate_Pct"].values[0])
}
with open("outputs/reports/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved outputs/reports/metrics.csv and summary.json!")


## 9. Milestone 6: Streamlit Advisor App (`app/app.py`)


In [ ]:
%%writefile app/app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import json

st.set_page_config(page_title="Airline Fare Intelligence & Advisor", page_icon="✈️", layout="wide")
st.title("✈️ Airline Fare Intelligence & Buy-Now-or-Wait Advisor")
st.markdown("Predict fair domestic airfares, 7-day surge risk, and get backtested booking advice.")

@st.cache_resource
def load_assets():
    fare_m = joblib.load("models/best_fare_lgb.pkl")
    clf_m = joblib.load("models/best_rise_classifier.pkl")
    with open("outputs/reports/summary.json") as f:
        meta = json.load(f)
    return fare_m, clf_m, meta

fare_model, clf_model, meta = load_assets()

TIME_ORDER = {"Early_Morning": 0, "Morning": 1, "Afternoon": 2, "Evening": 3, "Night": 4, "Late_Night": 5}

with st.sidebar:
    st.header("Search Flight")
    origin = st.selectbox("Origin", ["Delhi", "Mumbai", "Bangalore", "Kolkata", "Hyderabad", "Chennai"])
    destination = st.selectbox("Destination", ["Mumbai", "Delhi", "Bangalore", "Kolkata", "Hyderabad", "Chennai"], index=0)
    airline = st.selectbox("Airline", ["Indigo", "Vistara", "Air_India", "SpiceJet", "GO_FIRST", "AirAsia"])
    travel_class = st.selectbox("Class", ["Economy", "Business"])
    stops_num = st.selectbox("Stops", [0, 1, 2])
    dep_time = st.selectbox("Departure Window", list(TIME_ORDER.keys()), index=3)
    days_to_dep = st.slider("Days Left to Departure", 1, 49, 21)
    current_fare = st.number_input("Current Quoted Fare (₹)", min_value=1000, max_value=130000, value=5800, step=100)

if st.button("Analyze Fare & Get Advice", type="primary"):
    if origin == destination:
        st.error("Origin and Destination cannot be the same.")
    else:
        base_row = pd.DataFrame([{
            "days_to_departure": days_to_dep,
            "stops_num": stops_num,
            "duration_mins": 140 if stops_num == 0 else 360,
            "dep_time_idx": TIME_ORDER[dep_time],
            "arr_time_idx": (TIME_ORDER[dep_time] + 1) % 6,
            "is_red_eye": int(dep_time in ["Late_Night", "Early_Morning"]),
            "is_peak_evening": int(dep_time in ["Evening", "Night"]),
            "airline": airline,
            "origin": origin,
            "destination": destination,
            "travel_class": travel_class
        }])
        for c in ["airline", "origin", "destination", "travel_class"]:
            base_row[c] = base_row[c].astype("category")

        pred_fair = float(np.exp(fare_model.predict(base_row)[0]))
        next_row = base_row.copy()
        next_row["days_to_departure"] = max(1, days_to_dep - 7)
        pred_next = float(np.exp(fare_model.predict(next_row)[0]))

        clf_row = base_row.copy()
        clf_row["fare_inr"] = current_fare
        clf_row["pred_fair_fare"] = pred_fair
        clf_row["pred_next_week_fare"] = pred_next
        clf_row["fare_ratio_to_fair"] = current_fare / pred_fair
        clf_row["expected_move_pct"] = (pred_next - current_fare) / current_fare
        clf_row["lag_fare_change_pct"] = 0.0

        clf_cols = [
            "days_to_departure", "fare_inr", "pred_fair_fare", "pred_next_week_fare",
            "fare_ratio_to_fair", "expected_move_pct", "lag_fare_change_pct",
            "stops_num", "dep_time_idx", "arr_time_idx", "is_red_eye", "is_peak_evening",
            "airline", "origin", "destination", "travel_class"
        ]
        prob_rise = float(clf_model.predict_proba(clf_row[clf_cols])[0, 1])

        tau = meta.get("optimal_tau", 0.20)
        should_wait = (prob_rise < tau) and (pred_next < current_fare)

        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Current Fare", f"₹{current_fare:,.0f}")
        c2.metric("Predicted Fair Fare", f"₹{pred_fair:,.0f}", delta=f"{((current_fare-pred_fair)/pred_fair)*100:+.1f}% vs fair")
        c3.metric("7-Day >5% Rise Risk", f"{prob_rise*100:.1f}%")
        c4.metric("Advisor Decision", "⏳ WAIT" if should_wait else "🎫 BOOK NOW")

        if should_wait:
            st.success(f"**Recommendation: WAIT.** Surge probability is low ({prob_rise*100:.1f}% < {tau*100:.0f}%) and expected next-week fare is ₹{pred_next:,.0f}.")
        else:
            st.warning(f"**Recommendation: BOOK NOW.** High risk of fare rise ({prob_rise*100:.1f}%) or fare is already at/below expected next-week price (₹{pred_next:,.0f}).")
